In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [3]:
df=pd.read_csv("/kaggle/input/global-crocodile-species-dataset/crocodile_dataset.csv")

In [6]:
df.sample(3)

,Observation ID,Common Name,Scientific Name,Family,Genus,Observed Length (m),Observed Weight (kg),Age Class,Sex,Date of Observation,Country/Region,Habitat Type,Conservation Status,Observer Name,Notes
622,623,Saltwater Crocodile,Crocodylus porosus,Crocodylidae,Crocodylus,4.44,509.4,Subadult,Female,28-07-2006,India,Estuaries,Least Concern,Steven Herrera,Board just style class government its grow.
318,319,Borneo Crocodile (disputed),Crocodylus raninus,Crocodylidae,Crocodylus,2.81,138.5,Adult,Female,11-11-2012,Indonesia (Borneo),Estuarine Systems,Data Deficient,Rebecca Kelley,In night house clear say hope.
15,16,Morelet's Crocodile,Crocodylus moreletii,Crocodylidae,Crocodylus,2.00,67.9,Adult,Unknown,18-03-2007,Mexico,Rivers,Least Concern,Donald Wright,Kitchen technology nearly anything yourself st...


In [7]:
label_encode_value={}
label_encoder=LabelEncoder()
for i in df.columns:
    df[i]=label_encoder.fit_transform(df[i])
    label_encode_value[i]=label_encoder

In [8]:
df

,Observation ID,Common Name,Scientific Name,Family,Genus,Observed Length (m),Observed Weight (kg),Age Class,Sex,Date of Observation,Country/Region,Habitat Type,Conservation Status,Observer Name,Notes
0,0,7,5,0,0,139,221,0,1,917,1,27,3,19,141
1,1,0,0,0,0,334,718,0,1,813,45,16,4,109,28
2,2,11,2,0,0,66,448,2,2,212,45,6,0,656,219
3,3,7,5,0,0,190,343,0,1,22,29,22,3,287,568
4,4,8,8,0,0,313,691,0,2,435,17,22,4,269,161
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,995,15,13,0,0,203,427,0,0,18,28,14,3,498,885
996,996,6,1,0,0,175,300,3,0,719,34,11,3,895,602
997,997,17,14,0,1,182,354,0,1,176,11,27,0,556,666
998,998,17,14,0,1,230,541,0,1,262,37,23,0,311,680


In [9]:
X=df.drop(columns=["Conservation Status"])
y=df["Conservation Status"]

In [10]:
!pip install mindspore==2.7.1 -i https://repo.mindspore.cn/pypi/simple --trusted-host repo.mindspore.cn --extra-index-url https://repo.huaweicloud.com/repository/pypi/simple


Looking in indexes: https://repo.mindspore.cn/pypi/simple, https://repo.huaweicloud.com/repository/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 792.3/792.3 MB 1.4 MB/s eta 0:00:0000:0100:05


In [11]:
import mindspore
from mindspore import nn
from mindspore.dataset import vision, transforms
from mindspore.dataset import MnistDataset


In [13]:
!pip install download

In [14]:
# Download data from open datasets\n",
from download import download

url = "https://mindspore-website.obs.cn-north-4.myhuaweicloud.com/" \
      "notebook/datasets/MNIST_Data.zip"
path = download(url, "./", kind="zip", replace=True)



file_sizes: 100%|██████████████████████████| 10.8M/10.8M [00:00<00:00, 17.3MB/s]
Extracting zip file...
Successfully downloaded / unzipped to ./


In [15]:
train_dataset = MnistDataset('MNIST_Data/train')
test_dataset = MnistDataset('MNIST_Data/test')


In [16]:
print(train_dataset.get_col_names())


['image', 'label']


In [17]:
def datapipe(dataset, batch_size):
    image_transforms = [
        vision.Rescale(1.0 / 255.0, 0),
        vision.Normalize(mean=(0.1307,), std=(0.3081,)),
        vision.HWC2CHW()
    ]

    label_transform = transforms.TypeCast(mindspore.int32)

    dataset = dataset.map(image_transforms, 'image')
    dataset = dataset.map(label_transform, 'label')
    dataset = dataset.batch(batch_size)
    return dataset


In [18]:
train_dataset = datapipe(train_dataset, 64)
test_dataset = datapipe(test_dataset, 64)


In [19]:
for image, label in test_dataset.create_tuple_iterator():
    print(f"Shape of image [N, C, H, W]: {image.shape} {image.dtype}")
    print(f"Shape of label: {label.shape} {label.dtype}")
    break


Shape of image [N, C, H, W]: (64, 1, 28, 28) Float32
Shape of label: (64,) Int32


In [20]:
for data in test_dataset.create_dict_iterator():
    print(f"Shape of image [N, C, H, W]: {data['image'].shape} {data['image'].dtype}")
    print(f"Shape of label: {data['label'].shape} {data['label'].dtype}")
    break


Shape of image [N, C, H, W]: (64, 1, 28, 28) Float32
Shape of label: (64,) Int32


In [21]:
# Define model
class Network(nn.Cell):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.dense_relu_sequential = nn.SequentialCell(
            nn.Dense(28*28, 512),
            nn.ReLU(),
            nn.Dense(512, 512),
            nn.ReLU(),
            nn.Dense(512, 10)
        )

    def construct(self, x):
        x = self.flatten(x)
        logits = self.dense_relu_sequential(x)
        return logits

model = Network()
print(model)


Network(
  (flatten): Flatten()
  (dense_relu_sequential): SequentialCell(
    (0): Dense(input_channels=784, output_channels=512, has_bias=True)
    (1): ReLU()
    (2): Dense(input_channels=512, output_channels=512, has_bias=True)
    (3): ReLU()
    (4): Dense(input_channels=512, output_channels=10, has_bias=True)
  )
)


In [22]:
# Instantiate loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = nn.SGD(model.trainable_params(), 1e-2)

# 1. Define forward function
def forward_fn(data, label):
    logits = model(data)
    loss = loss_fn(logits, label)
    return loss, logits

# 2. Get gradient function
grad_fn = mindspore.value_and_grad(forward_fn, None, optimizer.parameters, has_aux=True)

# 3. Define function of one-step training
def train_step(data, label):
    (loss, _), grads = grad_fn(data, label)
    optimizer(grads)
    return loss

def train(model, dataset):
    size = dataset.get_dataset_size()
    model.set_train()
    for batch, (data, label) in enumerate(dataset.create_tuple_iterator()):
        loss = train_step(data, label)

        if batch % 100 == 0:
            loss, current = loss.asnumpy(), batch
            print(f"loss: {loss:>7f}  [{current:>3d}/{size:>3d}]")


In [23]:
def test(model, dataset, loss_fn):
    num_batches = dataset.get_dataset_size()
    model.set_train(False)
    total, test_loss, correct = 0, 0, 0
    for data, label in dataset.create_tuple_iterator():
        pred = model(data)
        total += len(data)
        test_loss += loss_fn(pred, label).asnumpy()
        correct += (pred.argmax(1) == label).asnumpy().sum()
    test_loss /= num_batches
    correct /= total
    print(f"Test: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


In [24]:
epochs = 3
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(model, train_dataset)
    test(model, test_dataset, loss_fn)
print("Done!")


Epoch 1
-------------------------------
loss: 2.301030  [  0/938]
loss: 1.680410  [100/938]
loss: 0.895080  [200/938]
loss: 0.667980  [300/938]
loss: 0.474997  [400/938]
loss: 0.390095  [500/938]
loss: 0.445638  [600/938]
loss: 0.301963  [700/938]
loss: 0.259552  [800/938]
loss: 0.270603  [900/938]
Test: 
 Accuracy: 90.8%, Avg loss: 0.323135 

Epoch 2
-------------------------------
loss: 0.387173  [  0/938]
loss: 0.204864  [100/938]
loss: 0.245484  [200/938]
loss: 0.300438  [300/938]
loss: 0.376621  [400/938]
loss: 0.324942  [500/938]
loss: 0.310968  [600/938]
loss: 0.341523  [700/938]
loss: 0.272387  [800/938]
loss: 0.250485  [900/938]
Test: 
 Accuracy: 92.6%, Avg loss: 0.253412 

Epoch 3
-------------------------------
loss: 0.293979  [  0/938]
loss: 0.137209  [100/938]
loss: 0.190364  [200/938]
loss: 0.181395  [300/938]
loss: 0.116473  [400/938]
loss: 0.246389  [500/938]
loss: 0.278128  [600/938]
loss: 0.261745  [700/938]
loss: 0.239117  [800/938]
loss: 0.184075  [900/938]
Test: 
 

In [25]:
# Save checkpoint
mindspore.save_checkpoint(model, "model.ckpt")
print("Saved Model to model.ckpt")


Saved Model to model.ckpt


In [26]:
# Instantiate a random initialized model
model = Network()
# Load checkpoint and load parameter to model
param_dict = mindspore.load_checkpoint("model.ckpt")
param_not_load, _ = mindspore.load_param_into_net(model, param_dict)
print(param_not_load)


[]


In [27]:
model.set_train(False)
for data, label in test_dataset:
    pred = model(data)
    predicted = pred.argmax(1)
    print(f'Predicted: "{predicted[:10]}", Actual: "{label[:10]}"')
    break


Predicted: "[2 2 4 3 3 7 1 5 0 3]", Actual: "[2 2 4 3 3 7 1 5 0 3]"
